In [ ]:
DATA = "bookings.jsonl"

In [2]:
import json
from datetime import date, datetime
from typing import Literal


import pandas as pd
from pydantic import BaseModel, Field, ValidationError, model_validator

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

In [3]:
raw_bookings = []     # (line_number, parsed dict) for lines that are valid JSON
quarantine = []       # every rejected line goes here, with a reason
lines_read = 0

with open(DATA, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():              # skip blank lines
            continue
        lines_read += 1
        try:
            doc = json.loads(line)
        except json.JSONDecodeError:
            quarantine.append({"booking_id": None, "line": line_no,
                               "reason": "invalid JSON"})
            continue
        if not isinstance(doc, dict):     # valid JSON but not an object
            quarantine.append({"booking_id": None, "line": line_no,
                               "reason": "not a JSON object"})
            continue
        raw_bookings.append((line_no, doc))

print(f"{lines_read} lines read")
print(f"{len(raw_bookings)} parsed, {len(quarantine)} quarantined so far")

283 lines read
283 parsed, 0 quarantined so far


In [4]:
class Student(BaseModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")
    name: str = Field(min_length=1)
    faculty: Literal["MET", "IET", "Management", "EMS","Pharmacy"]
   


In [5]:

class Ticket(BaseModel):
    type: Literal["standard", "vip", "student"]
    qty : int = Field(ge=1)          # ge: greater than or equal to
    price: float = Field(ge=0)
    

In [6]:

class Event(BaseModel):
    id: str = Field(min_length=1)
    title: str = Field(min_length=1)
    club: Literal["Robotics","Debate","Music","Football","Chess"]
    date: date
    venue: str = Field(min_length=1)

In [7]:
class Booking(BaseModel):
    booking_id: str = Field(min_length=1)
    booked_at: datetime     
    student:Student
    event:Event
    tickets: list[Ticket] = Field(min_length=1)
    status: Literal["confirmed", "cancelled"]   
    total: float
    payment: Literal["cash", "card", "wallet"]
    
   
    @model_validator(mode="after")
    def check_total(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if abs(self.total - expected) > 0.01:
            raise ValueError("total does not match sum of ticket prices")
        return self
       
    @model_validator(mode="after")
    def check_booking_date(self):
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked on or after the event date")
        return self

In [8]:
def make_reason(err):
    """One Pydantic error -> a short, stable text (no bad values in it)."""
    where = ".".join(str(p) for p in err["loc"])
    msg = err["msg"].removeprefix("Value error, ")
    return f"{where}: {msg}" if where else msg

valid = []   # (line_no, Booking)

for line_no, doc in raw_bookings:
    try:
        valid.append((line_no, Booking.model_validate(doc)))
    except ValidationError as e:
        errors = e.errors()
        quarantine.append({
            "booking_id": doc.get("booking_id"),
            "line": line_no,
            "reason": make_reason(errors[0]),               # main reason, used for counting
            "details": [make_reason(x) for x in errors],    # every problem
            "record": doc,                                  # the raw line, so it can be replayed
        })

print(f"{len(valid)} passed validation, {len(quarantine)} quarantined so far")

262 passed validation, 21 quarantined so far


In [9]:
seen = set()
bookings_ok = []   # valid, unique Booking objects

for line_no, booking in valid:
    if booking.booking_id in seen:
        quarantine.append({
            "booking_id": booking.booking_id,
            "line": line_no,
            "reason": "duplicate booking_id",
            "details": ["duplicate booking_id"],
            "record": booking.model_dump(mode="json"),
        })
    else:
        seen.add(booking.booking_id)
        bookings_ok.append(booking)

print(f"{len(bookings_ok)} bookings left, {len(quarantine)} quarantined in total")

259 bookings left, 24 quarantined in total


In [10]:
with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

In [11]:
from collections import Counter

reason_counts = Counter(q["reason"] for q in quarantine)

print(f"Lines read:   {lines_read}")
print(f"Passed:       {len(bookings_ok)}")
print(f"Quarantined:  {len(quarantine)}")
print("\nQuarantined by reason:")
for reason, n in reason_counts.most_common():
    print(f"  {n:>4}  {reason}")

# nothing lost: every line is either kept or quarantined
assert len(bookings_ok) + len(quarantine) == lines_read

Lines read:   283
Passed:       259
Quarantined:  24

Quarantined by reason:
     4  student.id: String should match pattern '^\d{2}-\d{4}$'
     4  total does not match sum of ticket prices
     3  tickets.0.type: Input should be 'standard', 'vip' or 'student'
     3  tickets.0.qty: Input should be greater than or equal to 1
     3  booked on or after the event date
     3  duplicate booking_id
     2  event: Input should be a valid dictionary or instance of Event
     2  tickets: List should have at least 1 item after validation, not 0


In [12]:
bookings_rows = []
ticket_rows = []

for b in bookings_ok:
    bookings_rows.append({
        "booking_id":    b.booking_id,
        "booked_at":     b.booked_at,
        "student_id":    b.student.id,
        "student_name":  b.student.name,
        "student_faculty": b.student.faculty,
        "event_id":      b.event.id,
        "event_title":   b.event.title,
        "event_club":    b.event.club,
        "event_date":    b.event.date,
        "event_venue":   b.event.venue,
        "total":         b.total,
        "payment":       b.payment,
        "status":        b.status,
    })
    for t in b.tickets:                       # a list becomes rows
        ticket_rows.append({
            "booking_id": b.booking_id,
            "type":       t.type,
            "qty":        t.qty,
            "price":      t.price,
            "amount":     t.qty * t.price,
        })

bookings = pd.DataFrame(bookings_rows)
booking_tickets = pd.DataFrame(ticket_rows)

print(bookings.shape, booking_tickets.shape)
bookings.head(3)

(259, 13) (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status
0,B0001,2026-09-28 15:46:00+00:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled
1,B0002,2026-09-27 12:29:00+00:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed
2,B0003,2026-10-13 20:19:00+00:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed


In [13]:
assert bookings["booking_id"].is_unique

sums = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")["total"]
assert ((sums - check).abs() < 0.01).all()

In [14]:
EXPECTED_TOP     = {"booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status"}
EXPECTED_STUDENT = {"id", "name", "faculty"}
EXPECTED_EVENT   = {"id", "title", "club", "date", "venue"}
EXPECTED_TICKET  = {"type", "qty", "price"}

rows = []   # (booked_at as a timestamp, set of extra fields in that document)

for line_no, doc in raw_bookings:
    extra = {k for k in doc if k not in EXPECTED_TOP}

    if isinstance(doc.get("student"), dict):
        extra |= {"student." + k for k in doc["student"] if k not in EXPECTED_STUDENT}
    if isinstance(doc.get("event"), dict):
        extra |= {"event." + k for k in doc["event"] if k not in EXPECTED_EVENT}
    if isinstance(doc.get("tickets"), list):
        for t in doc["tickets"]:
            if isinstance(t, dict):
                extra |= {"tickets." + k for k in t if k not in EXPECTED_TICKET}

    when = pd.to_datetime(doc.get("booked_at"), utc=True, errors="coerce")
    rows.append((when, extra))

all_extra = sorted(set().union(*[e for _, e in rows]))

if not all_extra:
    print("no extra fields found")

for field in all_extra:
    with_field    = [t for t, e in rows if field in e and pd.notna(t)]
    without_field = [t for t, e in rows if field not in e and pd.notna(t)]
    start = min(with_field)

    print(f"field: {field}")
    print(f"  documents that have it: {sum(field in e for _, e in rows)}")
    print(f"  first booked_at with it: {start}")
    print(f"  documents without it booked on or after that: {sum(t >= start for t in without_field)}")

field: promo_code
  documents that have it: 193
  first booked_at with it: 2026-10-20 01:05:00+00:00
  documents without it booked on or after that: 1


In [15]:
import duckdb
con = duckdb.connect()                    # in memory: fresh on every run
con.sql("set TimeZone = 'UTC'")           # date_key must follow the UTC booked_at

con.sql("create or replace table bookings as select * from bookings")
con.sql("create or replace table booking_tickets as select * from booking_tickets")

con.sql("""
    create or replace table dim_event as
    select row_number() over (order by event_id) as event_key,
           event_id, event_title as title, event_club as club,
           event_date as date, event_venue as venue
    from (select * from bookings
          qualify row_number() over (partition by event_id order by booked_at) = 1)
""")

con.sql("""
    create or replace table dim_student as
    select row_number() over (order by student_id) as student_key,
           student_id, student_name as name, student_faculty as faculty
    from (select * from bookings
          qualify row_number() over (partition by student_id order by booked_at) = 1)
""")

con.sql("""
    create or replace table fact_ticket_sales as
    select b.booking_id,
           e.event_key,
           s.student_key,
           t.type as ticket_type,
           t.qty, t.price, t.amount,
           b.status,
           strftime(b.booked_at, '%Y%m%d')::int as date_key
    from booking_tickets t
    join bookings b    on b.booking_id = t.booking_id
    join dim_student s on s.student_id = b.student_id
    join dim_event e   on e.event_id   = b.event_id
""")

CountBIGINT
314


In [16]:
n_fact = con.sql("select count(*) from fact_ticket_sales").fetchall()[0][0]
assert n_fact == len(booking_tickets), (n_fact, len(booking_tickets))

fact_sum = con.sql("select sum(amount) from fact_ticket_sales").fetchall()[0][0]
assert abs(fact_sum - bookings["total"].sum()) < 0.01, (fact_sum, bookings["total"].sum())

print("fact rows:", n_fact, "· sum of amount:", fact_sum)

fact rows: 314 · sum of amount: 31285.0


In [17]:
answer = con.sql("""
    select e.club,
           f.ticket_type,
           sum(f.qty)    as tickets_sold,
           sum(f.amount) as revenue
    from fact_ticket_sales f
    join dim_event e on e.event_key = f.event_key
    where f.status = 'confirmed'
    group by e.club, f.ticket_type
    order by e.club, f.ticket_type
""").df()
answer

,club,ticket_type,tickets_sold,revenue
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


In [18]:
check = (
    booking_tickets.merge(bookings[["booking_id", "event_club", "status"]], on="booking_id")
    .query("status == 'confirmed'")
    .groupby(["event_club", "type"], as_index=False)
    .agg(tickets_sold=("qty", "sum"), revenue=("amount", "sum"))
    .sort_values(["event_club", "type"], ignore_index=True)
)

assert len(check) == len(answer)
assert (check["tickets_sold"].values == answer["tickets_sold"].values).all()
assert abs(check["revenue"].sum() - answer["revenue"].sum()) < 0.01
print("star answer matches pandas")

star answer matches pandas


In [19]:
import shutil
from pathlib import Path

STAR = Path("star")
shutil.rmtree(STAR, ignore_errors=True)   # start clean, so a re-run (or the hidden file) leaves no old folders
STAR.mkdir()

# each dimension: one file
con.execute("copy dim_event   to 'star/dim_event.parquet'   (format parquet)")
con.execute("copy dim_student to 'star/dim_student.parquet' (format parquet)")

# the fact: one folder per booking month
con.execute("""
    copy (
        select *,
               strftime(strptime(date_key::varchar, '%Y%m%d'), '%Y-%m') as booked_month
        from fact_ticket_sales
    ) to 'star/fact_ticket_sales' (format parquet, partition_by (booked_month))
""")

for p in sorted(STAR.rglob("*.parquet")):
    print(p.as_posix())

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet


In [20]:
seen_ids, swap_dups, swap_kept = set(), 0, []
for line_no, doc in raw_bookings:
    bid = doc.get("booking_id")
    if bid is not None and bid in seen_ids:
        swap_dups += 1
    else:
        seen_ids.add(bid)
        swap_kept.append(doc)

swap_ok, swap_bad = 0, Counter()
for doc in swap_kept:
    try:
        Booking.model_validate(doc)
        swap_ok += 1
    except ValidationError as e:
        swap_bad[make_reason(e.errors()[0])] += 1

print("duplicates:", swap_dups, "| bookings left:", swap_ok, "| validation failures:", sum(swap_bad.values()))

duplicates: 3 | bookings left: 259 | validation failures: 21


In [21]:
con.sql("""
    select sum(b.total) from fact_ticket_sales f
    join bookings b using (booking_id)
""").fetchall()[0][0]

42000.0